# Hidden State Extraction

Extracts hidden states from Llama-3.1-8B-Instruct for one conversation folder at a time.

**Design:**
- Layer: last transformer layer (`hidden_states[-1]`)
- Position: last token of each assistant turn
- One forward pass per conversation; turn positions found by tokenizing prefixes
- Crescendo: refusal turns (logged via `is_refusal` field) excluded from context

**Output per folder:**
- `data/representations/{FOLDER_NAME}/hidden_states.npy` — float16, shape (N, 4096)
- `data/representations/{FOLDER_NAME}/metadata.parquet` — one row per (conversation × turn)

Run this notebook once per folder. Change `FOLDER_NAME` in the config cell and re-run.

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "4"

import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM

repo_root = Path("..").resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

# ── Config ────────────────────────────────────────────────────────────────────
FOLDER_NAME = "actorattack_harmful_v2"   # ← change this to process a different folder

MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"
DEVICE   = "cuda:0"   # cuda:0 because CUDA_VISIBLE_DEVICES=4
DTYPE    = torch.bfloat16

CONV_DIR = repo_root / "data" / "conversations" / FOLDER_NAME
SAVE_DIR = repo_root / "data" / "representations" / FOLDER_NAME
SAVE_DIR.mkdir(parents=True, exist_ok=True)

STATES_PATH   = SAVE_DIR / "hidden_states.npy"
METADATA_PATH = SAVE_DIR / "metadata.parquet"

# Derive framework and split from folder name convention: {framework}_{split}_v2
parts     = FOLDER_NAME.split("_")
FRAMEWORK = parts[0]                    # e.g. "actorattack"
SPLIT     = parts[1]                    # e.g. "harmful" or "benign"

assert CONV_DIR.exists(), f"Conversation folder not found: {CONV_DIR}"

print(f"Folder:    {FOLDER_NAME}")
print(f"Framework: {FRAMEWORK}")
print(f"Split:     {SPLIT}")
print(f"Input:     {CONV_DIR}")
print(f"Output:    {SAVE_DIR}")

In [ ]:
print(f"Loading {MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=DTYPE,
    device_map=DEVICE,
)
model.eval()
print("Model loaded.")
print(f"  Layers:     {model.config.num_hidden_layers}")
print(f"  Hidden dim: {model.config.hidden_size}")

In [ ]:
def build_context(turns: list[dict], framework: str) -> tuple[list[dict], list[dict]]:
    """
    Build messages list and list of assistant turns to extract at.

    For Crescendo: skips turns where is_refusal=True (logged during generation).
    These turns were rolled back from the target's context, so excluding them
    gives a fresh forward pass that matches generation conditions exactly.

    For other frameworks: all turns used as-is.
    """
    messages = []
    extract_at = []

    by_idx = {}
    for t in turns:
        by_idx.setdefault(t["turn_idx"], {})[t["role"]] = t

    for turn_idx in sorted(by_idx):
        pair = by_idx[turn_idx]
        user_turn = pair.get("user")
        asst_turn = pair.get("assistant")
        if not user_turn or not asst_turn:
            continue

        if framework == "crescendo" and asst_turn.get("is_refusal", False):
            continue

        messages.append({"role": "user",      "content": user_turn["content"]})
        messages.append({"role": "assistant", "content": asst_turn["content"]})
        extract_at.append({"turn_idx": turn_idx, "n_messages": len(messages)})

    return messages, extract_at


def find_turn_end_positions(tokenizer, messages: list[dict], extract_at: list[dict]) -> list[int]:
    """
    For each assistant turn in extract_at, find the index of its last token
    in the full tokenized sequence using prefix tokenization.
    """
    positions = []
    for entry in extract_at:
        prefix = messages[:entry["n_messages"]]
        ids = tokenizer.apply_chat_template(
            prefix,
            return_tensors="pt",
            add_generation_prompt=False,
        )
        positions.append(ids.shape[1] - 1)
    return positions


@torch.no_grad()
def extract_hidden_states(model, tokenizer, messages: list[dict], positions: list[int]) -> np.ndarray:
    """
    One forward pass on the full conversation.
    Returns array of shape (len(positions), hidden_dim) in float16.
    """
    input_ids = tokenizer.apply_chat_template(
        messages,
        return_tensors="pt",
        add_generation_prompt=False,
    ).to(model.device)

    outputs = model(input_ids, output_hidden_states=True)
    last_hidden = outputs.hidden_states[-1][0]  # (seq_len, hidden_dim)

    vecs = [last_hidden[pos].cpu().to(torch.float16).numpy() for pos in positions]
    return np.stack(vecs, axis=0)  # (n_turns, hidden_dim)


print("Helpers defined.")

In [ ]:
# ── Extraction ────────────────────────────────────────────────────────────────
# Resume support: skip already-processed (pair_id, attempt) pairs
already_done = set()
if METADATA_PATH.exists():
    existing_meta = pd.read_parquet(METADATA_PATH)
    already_done = set(zip(existing_meta["pair_id"], existing_meta["attempt"]))
    print(f"Resuming: {len(existing_meta)} rows already extracted ({len(already_done)} conversations)")
    all_states = [np.load(str(STATES_PATH))]
    all_meta   = [existing_meta]
else:
    all_states = []
    all_meta   = []
    print("Starting fresh extraction.")

files = sorted(CONV_DIR.glob("*.json"))
print(f"Files to process: {len(files)}")

for fpath in tqdm(files, desc=FOLDER_NAME):
    conv    = json.loads(fpath.read_text())
    pair_id = conv["objective_pair_id"]
    attempt = conv.get("attempt", 1)
    verdict = conv["verdict"]

    if (pair_id, attempt) in already_done:
        continue

    turns = conv.get("turns", [])
    if not turns:
        continue

    try:
        messages, extract_at = build_context(turns, FRAMEWORK)
        if not extract_at:
            continue

        positions = find_turn_end_positions(tokenizer, messages, extract_at)
        states    = extract_hidden_states(model, tokenizer, messages, positions)

    except Exception as e:
        tqdm.write(f"  ERROR {fpath.name}: {e}")
        continue

    rows = [
        {
            "framework": FRAMEWORK,
            "split":     SPLIT,
            "pair_id":   pair_id,
            "attempt":   attempt,
            "turn_idx":  entry["turn_idx"],
            "verdict":   verdict,
            "label":     1 if SPLIT == "harmful" else 0,
            "fname":     fpath.name,
        }
        for entry in extract_at
    ]

    all_states.append(states)
    all_meta.append(pd.DataFrame(rows))
    already_done.add((pair_id, attempt))

print("\nSaving...")
final_states = np.concatenate(all_states, axis=0).astype(np.float16)
final_meta   = pd.concat(all_meta, ignore_index=True)

np.save(str(STATES_PATH), final_states)
final_meta.to_parquet(METADATA_PATH, index=False)

print(f"Saved {final_states.shape[0]} vectors → {STATES_PATH}")
print(f"  Shape: {final_states.shape}  ({final_states.nbytes / 1e6:.1f} MB)")
final_meta.groupby(["split", "verdict"])["turn_idx"].count().rename("n_vectors").reset_index()

## Verification

In [ ]:
states = np.load(str(STATES_PATH))
meta   = pd.read_parquet(METADATA_PATH)

print(f"States shape: {states.shape}")
print(f"Metadata rows: {len(meta)}")
print()
print("Vectors per verdict:")
display(meta.groupby(["split", "verdict"])["turn_idx"].count().rename("n_vectors"))
print()
print("Vectors per turn position:")
display(meta.groupby("turn_idx")["pair_id"].count().rename("n_vectors"))
print()
sample = states[np.random.choice(len(states), min(1000, len(states)), replace=False)].astype(np.float32)
print(f"State stats: mean={sample.mean():.4f}  std={sample.std():.4f}  NaNs={np.isnan(sample).sum()}  Infs={np.isinf(sample).sum()}")